<a href="https://colab.research.google.com/github/BhargaviU2004/CalmQuest/blob/main/TableTalk_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install librosa pandas numpy scikit-learn soundfile openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=44cfb58fa0c2aea05d15a2d3f3e2b0c06632b4c131901ae3bd8b2ce84d9de91e
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [9]:
from google.colab import drive
import librosa
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
drive.mount('/content/drive')
data_dir = '/content/drive/MyDrive/Voice'
target_actors = ['Actor_01', 'Actor_02', 'Actor_03']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
def extract_features(file_path):
    # Task 1
    y, sr = librosa.load(file_path, mono=True)
    y = librosa.util.normalize(y)

    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).T, axis=0)

    pitches = librosa.yin(y, fmin=75, fmax=300)
    pitch_mean = np.mean(pitches)

    energy = np.mean(librosa.feature.rms(y=y))
    duration = librosa.get_duration(y=y, sr=sr)

    return list(mfccs) + [pitch_mean, energy, duration]

def get_tone_label(filename):
    code = filename.split('-')[2]
    mapping = {
        "01": "neutral",
        "02": "calm description",
        "05": "urgency",
        "06": "suspense",
        "08": "dramatic emphasis"
    }
    return mapping.get(code, "other")

In [11]:
all_data = []

for actor in target_actors:
    actor_path = os.path.join(data_dir, actor)
    if os.path.exists(actor_path):
        files = [f for f in os.listdir(actor_path) if f.endswith('.wav')]
        print(f"Processing {len(files)} files from {actor}...")

        for filename in files:
            file_path = os.path.join(actor_path, filename)
            try:
                features = extract_features(file_path)
                tone = get_tone_label(filename)
                all_data.append([filename, actor, tone] + features)
            except Exception as e:
                print(f"Skipping {filename}: {e}")

cols = ['filename', 'actor', 'tone', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5',
        'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13',
        'pitch', 'energy', 'duration']

df = pd.DataFrame(all_data, columns=cols)
print(f"\nTask 1 Complete: {len(df)} recordings processed.")
df.head()

Processing 44 files from Actor_01...
Processing 44 files from Actor_02...
Processing 44 files from Actor_03...

Task 1 Complete: 132 recordings processed.


,filename,actor,tone,mfcc1,mfcc2,mfcc3,mfcc4,mfcc5,mfcc6,mfcc7,mfcc8,mfcc9,mfcc10,mfcc11,mfcc12,mfcc13,pitch,energy,duration
0,03-02-03-02-01-02-01.wav,Actor_01,other,-338.072449,46.202354,-19.161522,10.266398,-2.487244,-8.333303,-17.382942,-5.853585,-20.311510,-9.100236,0.472553,-12.570916,-2.788753,163.750703,0.092062,4.804807
1,03-02-03-02-02-01-01.wav,Actor_01,other,-318.817352,44.221947,-23.751169,9.087063,-5.701321,-8.484038,-21.735100,-6.596186,-19.839092,-8.206760,-1.430951,-10.099604,-2.321979,164.499913,0.093517,4.938277
2,03-02-02-01-02-01-01.wav,Actor_01,calm description,-312.663666,58.501701,-1.819185,12.024393,-1.849748,-1.259125,-9.172065,-3.576785,-18.590498,-7.300974,-0.730215,-5.259278,-6.762022,152.462888,0.136180,4.504535
3,03-02-02-02-01-01-01.wav,Actor_01,calm description,-313.862488,66.449493,-14.225901,12.998992,1.618998,-5.531011,-12.190064,-8.542605,-18.021889,-0.236985,0.598892,-7.345278,-3.324453,172.842778,0.143143,4.704717
4,03-02-04-01-02-02-01.wav,Actor_01,other,-315.297668,54.642899,-12.577813,10.267268,-2.378792,-5.390404,-8.305521,-10.356411,-20.050936,-6.694321,-1.810557,-7.454474,-4.242743,158.206212,0.117668,4.571293


In [12]:

!pip install openai-whisper

import whisper
import os

model_whisper = whisper.load_model("base")

In [15]:
transcripts = []

for index, row in df.iterrows():

    file_path = os.path.join(data_dir, row['actor'], row['filename'])

    try:
        result = model_whisper.transcribe(file_path)
        text = result['text'].strip()
        transcripts.append(text)

        if (index + 1) % 10 == 0:
            print(f"Processed {index + 1}/{len(df)} transcripts...")

    except Exception as e:
        print(f"Error transcribing {row['filename']}: {e}")
        transcripts.append("TRANSCRIPTION_ERROR")

df['transcript'] = transcripts

print("\nTask 3 Complete! Sample Transcripts:")
print(df[['filename', 'tone', 'transcript']].head())

Processed 10/132 transcripts...
Processed 20/132 transcripts...
Processed 30/132 transcripts...
Processed 40/132 transcripts...
Processed 50/132 transcripts...
Processed 60/132 transcripts...
Processed 70/132 transcripts...
Processed 80/132 transcripts...
Processed 90/132 transcripts...
Processed 100/132 transcripts...
Processed 110/132 transcripts...
Processed 120/132 transcripts...
Processed 130/132 transcripts...

Task 3 Complete! Sample Transcripts:
                   filename              tone                     transcript
0  03-02-03-02-01-02-01.wav             other  Kids are talking by the door.
1  03-02-03-02-02-01-01.wav             other  Dogs are sitting by the door.
2  03-02-02-01-02-01-01.wav  calm description  Dogs are sitting by the door.
3  03-02-02-02-01-01-01.wav  calm description  Kids are talking by the door.
4  03-02-04-01-02-02-01.wav             other  Dogs are sitting by the door.


In [16]:
df.to_csv('TableTalk_Technical_Test_Results.csv', index=False)
print("Results saved to TableTalk_Technical_Test_Results.csv")

from google.colab import files
files.download('TableTalk_Technical_Test_Results.csv')

Results saved to TableTalk_Technical_Test_Results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
def retrieve_narrative_audio(dataframe, tone=None, keyword=None, min_duration=0):

    results = dataframe.copy()
    if tone:
        results = results[results['tone'].str.contains(tone, case=False)]

    if keyword:
        results = results[results['transcript'].str.contains(keyword, case=False)]

    results = results[results['duration'] >= min_duration]

    return results[['filename', 'actor', 'tone', 'duration', 'transcript']]

print("--- SEARCH 1: Urgent clips about 'dogs' ---")
search_1 = retrieve_narrative_audio(df, tone="urgency", keyword="dog")
print(search_1 if not search_1.empty else "No matching clips found.")

print("\n--- SEARCH 2: Calm descriptions longer than 4 seconds ---")
search_2 = retrieve_narrative_audio(df, tone="calm", min_duration=4.0)
print(search_2.head() if not search_2.empty else "No matching clips found.")

--- SEARCH 1: Urgent clips about 'dogs' ---
                     filename     actor     tone  duration  \
6    03-02-05-02-02-02-01.wav  Actor_01  urgency  5.205215   
21   03-02-05-02-02-01-01.wav  Actor_01  urgency  4.971655   
24   03-02-05-01-02-02-01.wav  Actor_01  urgency  4.738095   
43   03-02-05-01-02-01-01.wav  Actor_01  urgency  4.604626   
68   03-02-05-01-02-02-02.wav  Actor_02  urgency  4.738095   
70   03-02-05-02-02-02-02.wav  Actor_02  urgency  4.537868   
71   03-02-05-02-02-01-02.wav  Actor_02  urgency  4.638005   
83   03-02-05-01-02-01-02.wav  Actor_02  urgency  4.738095   
115  03-02-05-02-02-02-03.wav  Actor_03  urgency  4.471156   
116  03-02-05-01-02-01-03.wav  Actor_03  urgency  4.871565   
119  03-02-05-02-02-01-03.wav  Actor_03  urgency  4.571293   
129  03-02-05-01-02-02-03.wav  Actor_03  urgency  4.537868   

                        transcript  
6    Dogs are sitting by the door.  
21   Dogs are sitting by the door.  
24   Dogs are sitting by the door.  
4